# NeuroFlex CUDA validation
Run this notebook on a Google Colab GPU runtime. It installs the repository, verifies CUDA, runs business-logic tests, and benchmarks tensor transfer/angle computation. This is engineering validation—not clinical validation.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

In [ ]:
# Upload the repository zip or replace this with your Git URL.
# !git clone https://github.com/YOUR_ACCOUNT/NeuroFlex1.git
# %cd NeuroFlex1
!pip install -e '.[cuda,pose]' pytest hypothesis

In [ ]:
!pytest -q

In [ ]:
import time, torch
device=torch.device('cuda')
x=torch.rand((10000,33,3),device=device)
torch.cuda.synchronize(); start=time.perf_counter()
a=x[:,0]-x[:,1]; b=x[:,2]-x[:,1]
angles=torch.rad2deg(torch.acos(torch.clamp((a*b).sum(-1)/(a.norm(dim=-1)*b.norm(dim=-1)),-1,1)))
torch.cuda.synchronize(); elapsed=(time.perf_counter()-start)*1000
print(f'10k vectorized angles: {elapsed:.2f} ms; mean={angles.mean().item():.2f}')